<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module1_Labs(v2)/Lab2_Qubits_BlochSphere_Hadamard.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 2 — Qubits, the Bloch Sphere & the Hadamard Gate
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Represent a qubit as a vector and as a point on the Bloch sphere
2. Understand what **amplitude** and **probability** mean
3. Visualize how the **Hadamard gate** (H) and **Pauli gates** (X, Y, Z) move a qubit on the Bloch sphere
4. Understand **superposition**: why H|0⟩ is not just "random" but a precise quantum state
5. Connect the Bloch sphere to QAOA Step 1: *Everyone on Stage*

---
### 📖 Background: The Qubit State

A classical bit is **0 or 1**. A qubit can be in a **superposition**:
$$|\psi\rangle = a_0|0\rangle + a_1|1\rangle = \begin{bmatrix} a_0 \\ a_1 \end{bmatrix}$$

where $a_0, a_1$ are **amplitudes** (complex numbers) with $|a_0|^2 + |a_1|^2 = 1$.

When measured:
- Probability of getting **0**: $P(0) = |a_0|^2$
- Probability of getting **1**: $P(1) = |a_1|^2$

The **Bloch sphere** is a unit sphere where every point on the surface represents a valid qubit state:
$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$$

- **North pole** ($\theta=0$): $|0\rangle$
- **South pole** ($\theta=\pi$): $|1\rangle$
- **Equator** ($\theta=\pi/2$): equal superposition, phase depends on $\phi$

---

In [ ]:
# Setup — run this cell first
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()
print("Setup complete.")

## Part 1: Building a Bloch Sphere Visualizer

We'll build our own Bloch sphere plotter to understand the geometry.

In [ ]:
def draw_bloch_sphere(states_list, labels=None, title="Bloch Sphere"):
    """
    Draw one or more qubit states on the Bloch sphere.
    states_list: list of [a0, a1] amplitude pairs (real or complex)
    labels:      list of strings for the legend
    """
    fig = plt.figure(figsize=(6, 6))
    ax  = fig.add_subplot(111, projection='3d')

    # Draw sphere wireframe
    u = np.linspace(0, 2*np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(xs, ys, zs, color='lightblue', alpha=0.08)

    # Draw axes
    for start, end, lbl, pos in [
        ([0,0,-1.4],[0,0,1.4],'z',[0,0,1.5]),
        ([-1.4,0,0],[1.4,0,0],'x',[1.5,0,0]),
        ([0,-1.4,0],[0,1.4,0],'y',[0,1.5,0]),
    ]:
        ax.plot([start[0],end[0]],[start[1],end[1]],[start[2],end[2]],'k-',lw=0.8)
        ax.text(*pos, lbl, fontsize=12, ha='center')

    # Label poles
    ax.text(0, 0, 1.15, '|0⟩', fontsize=11, ha='center', color='darkgreen')
    ax.text(0, 0,-1.15, '|1⟩', fontsize=11, ha='center', color='darkgreen')
    ax.text(1.1,0, 0,   '|+⟩', fontsize=10, ha='center', color='gray')

    colors = ['red','blue','purple','orange','green']

    for idx, state in enumerate(states_list):
        a0 = complex(state[0])
        a1 = complex(state[1])
        norm = np.sqrt(abs(a0)**2 + abs(a1)**2)
        if norm > 0:
            a0 /= norm; a1 /= norm

        # Convert to Bloch vector
        bx = 2 * np.real(np.conj(a0) * a1)
        by = 2 * np.imag(np.conj(a0) * a1)
        bz = abs(a0)**2 - abs(a1)**2

        color = colors[idx % len(colors)]
        lbl   = labels[idx] if labels else f'state {idx}'
        ax.quiver(0,0,0, bx,by,bz, color=color, lw=2.5, arrow_length_ratio=0.12, label=lbl)
        ax.scatter([bx],[by],[bz], color=color, s=60, zorder=5)

    ax.set_xlim([-1.3,1.3]); ax.set_ylim([-1.3,1.3]); ax.set_zlim([-1.3,1.3])
    ax.set_axis_off()
    ax.set_title(title, fontsize=13, pad=10)
    if labels:
        ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

print("Bloch sphere helper ready.")

In [ ]:
# ── 1.1  Basis states on the Bloch sphere ────────────────────────────────────
draw_bloch_sphere(
    states_list=[[1,0], [0,1]],
    labels=['|0⟩ (north pole)', '|1⟩ (south pole)'],
    title='Basis States on the Bloch Sphere'
)
print("  |0⟩ points UP (north pole)")
print("  |1⟩ points DOWN (south pole)")

In [ ]:
# ── 1.2  Superposition states on the equator ─────────────────────────────────
inv_sqrt2 = 1/np.sqrt(2)

# H|0⟩ = (|0⟩ + |1⟩)/√2  → points along +x axis
plus  = [inv_sqrt2,  inv_sqrt2]   # |+⟩
minus = [inv_sqrt2, -inv_sqrt2]   # |−⟩ = H|1⟩

draw_bloch_sphere(
    states_list=[plus, minus, [1,0], [0,1]],
    labels=['H|0⟩ = |+⟩ (+x axis)', 'H|1⟩ = |−⟩ (−x axis)',
            '|0⟩ (north)', '|1⟩ (south)'],
    title='Superposition States — H gate moves to the equator'
)
print("Key insight: H gate rotates |0⟩ from the north pole to the +x equator.")
print("Both |+⟩ and |−⟩ give P(0)=P(1)=0.5, but they are DIFFERENT quantum states!")
print("The difference is the PHASE — invisible at measurement but critical for interference.")

In [ ]:
# ── 1.3  Amplitude vs Probability ────────────────────────────────────────────
states = {
    '|0⟩':    np.array([1.0, 0.0]),
    '|1⟩':    np.array([0.0, 1.0]),
    'H|0⟩':   np.array([inv_sqrt2, inv_sqrt2]),
    'H|1⟩':   np.array([inv_sqrt2, -inv_sqrt2]),
}

print(f"{'State':<10} {'a₀':>10} {'a₁':>10} {'P(|0⟩)':>10} {'P(|1⟩)':>10}")
print("-" * 55)
for name, s in states.items():
    a0, a1 = s[0], s[1]
    print(f"{name:<10} {a0:>10.4f} {a1:>10.4f} {a0**2:>10.4f} {a1**2:>10.4f}")

print("\n⚠️  H|0⟩ and H|1⟩ have identical probabilities but different amplitudes (phases).")
print("    This phase difference drives quantum interference in QAOA.")

---
## Part 2: Pauli Gates on the Bloch Sphere

The three Pauli gates rotate a qubit by 180° around the X, Y, or Z axis.

In [ ]:
import numpy as np

I = np.eye(2)
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])
H = (1/np.sqrt(2)) * np.array([[1,1],[1,-1]])

ket_0 = np.array([1.0, 0.0])
ket_1 = np.array([0.0, 1.0])
plus  = (1/np.sqrt(2)) * np.array([1.0, 1.0])

print("Pauli gate actions:")
print(f"  X|0⟩ = {X @ ket_0}  (flips to |1⟩ — quantum NOT)")
print(f"  X|1⟩ = {X @ ket_1}  (flips to |0⟩)")
print(f"  Z|0⟩ = {Z @ ket_0}  (no change — |0⟩ is Z eigenstate)")
print(f"  Z|1⟩ = {Z @ ket_1}  (sign flip of |1⟩)")
print(f"  Z|+⟩ = {np.round(Z @ plus, 4)}  (|+⟩ → |−⟩: phase flip!)")

In [ ]:
# ── 2.1  Verify Pauli gates with Qiskit ──────────────────────────────────────
def apply_and_show(gate_name, gate_qiskit_fn, input_state_circuit=None):
    """Build a circuit, apply a gate, and return the statevector."""
    qc = QuantumCircuit(1)
    if input_state_circuit:
        qc.compose(input_state_circuit, inplace=True)
    gate_qiskit_fn(qc)
    sv = Statevector(qc)
    return sv

# X|0⟩
qc_x = QuantumCircuit(1)
qc_x.x(0)
sv_x = Statevector(qc_x)
print("X|0⟩ statevector:", sv_x)
print("X|0⟩ probabilities:", sv_x.probabilities_dict())

# Z(H|0⟩)
qc_zh = QuantumCircuit(1)
qc_zh.h(0)
qc_zh.z(0)
sv_zh = Statevector(qc_zh)
print("\nZ(H|0⟩) statevector:", np.round(sv_zh.data, 4))
print("Z(H|0⟩) probabilities:", sv_zh.probabilities_dict())

In [ ]:
# ── 2.2  Visualize Pauli-Z effect on H|0⟩ ────────────────────────────────────
before_z = np.array([1/np.sqrt(2),  1/np.sqrt(2)])   # H|0⟩ = |+⟩
after_z  = np.array([1/np.sqrt(2), -1/np.sqrt(2)])   # Z H|0⟩ = |−⟩

draw_bloch_sphere(
    states_list=[before_z, after_z],
    labels=['H|0⟩ = |+⟩ (before Z)', 'Z·H|0⟩ = |−⟩ (after Z)'],
    title='Pauli-Z: 180° rotation around z-axis'
)
print("Key: Z flips the sign of the |1⟩ component.")
print("On the Bloch sphere: 180° rotation around the z-axis.")
print("Both states have P(0)=P(1)=0.5, but opposite phases!")

### ✏️ Exercise 2.1 — Pauli Gate Exploration

For each of the following, compute the result **both** manually (NumPy) and with Qiskit's `Statevector`:

1. `X(H|0⟩)` — apply H then X. What state is this?
2. `H(X|0⟩)` — apply X then H. Is it the same as above?
3. `Z(Z|+⟩)` — apply Z twice to |+⟩. What do you get? Why?

In [ ]:
# YOUR CODE HERE
print("=== Exercise 2.1 ===")

# 1. X(H|0⟩)
result1_numpy = X @ (H @ ket_0)
qc1 = QuantumCircuit(1); qc1.h(0); qc1.x(0)
result1_qiskit = Statevector(qc1).data
print(f"1. X(H|0⟩)  NumPy:  {np.round(result1_numpy,4)}")
print(f"   X(H|0⟩)  Qiskit: {np.round(result1_qiskit,4)}")

# 2. H(X|0⟩)
result2_numpy = H @ (X @ ket_0)
qc2 = QuantumCircuit(1); qc2.x(0); qc2.h(0)
result2_qiskit = Statevector(qc2).data
print(f"\n2. H(X|0⟩)  NumPy:  {np.round(result2_numpy,4)}")
print(f"   H(X|0⟩)  Qiskit: {np.round(result2_qiskit,4)}")
print(f"   Same as X(H|0⟩)? {np.allclose(result1_numpy, result2_numpy)}")

# 3. Z(Z|+⟩)
result3_numpy = Z @ (Z @ plus)
qc3 = QuantumCircuit(1); qc3.h(0); qc3.z(0); qc3.z(0)
result3_qiskit = Statevector(qc3).data
print(f"\n3. Z(Z|+⟩)  NumPy:  {np.round(result3_numpy,4)}")
print(f"   Z(Z|+⟩)  Qiskit: {np.round(result3_qiskit,4)}")
print("   Z² = I: applying Z twice returns to the original state.")

---
## Part 3: The Hadamard Gate — Gateway to Superposition

H is the most important gate in QAOA Step 1. Let's understand it deeply.

In [ ]:
# ── 3.1  H² = I: Hadamard is its own inverse ──────────────────────────────────
H_squared = H @ H
print("H² =")
print(np.round(H_squared, 4))
print("\nH² = I: Applying H twice returns to the original state!")

# Verify in Qiskit
qc_hh = QuantumCircuit(1)
qc_hh.h(0)
qc_hh.h(0)   # second H undoes the first
sv_hh = Statevector(qc_hh)
print("\nQiskit H·H|0⟩ statevector:", sv_hh.data)  # should be [1, 0] = |0⟩

In [ ]:
# ── 3.2  H on n qubits creates uniform superposition ─────────────────────────
for n in [1, 2, 3]:
    qc = QuantumCircuit(n)
    qc.h(range(n))
    sv = Statevector(qc)
    probs = sv.probabilities_dict()
    expected_prob = 1 / 2**n
    print(f"\nn={n} qubits: {2**n} states, each with prob ≈ {expected_prob:.4f}")
    for bs, p in sorted(probs.items()):
        print(f"  |{bs}⟩: {p:.4f}")

print("\nFor 5 qubits (QAOA): 32 states, each with prob = 1/32 ≈ 0.03125")
print("This is the starting point: ALL candidate solutions are equally probable.")

In [ ]:
# ── 3.3  Visualize the effect of repeated H gates on the Bloch sphere ────────
states_trace = [
    [1, 0],                              # |0⟩  start
    [1/np.sqrt(2), 1/np.sqrt(2)],        # H|0⟩
    [1, 0],                              # H²|0⟩ = |0⟩ again
]
labels_trace = ['Step 0: |0⟩', 'Step 1: H|0⟩ = |+⟩', 'Step 2: H²|0⟩ = |0⟩']

draw_bloch_sphere(
    states_list=states_trace,
    labels=labels_trace,
    title='H gate: north pole ↔ equator (+x)'
)

---
## Part 4: QAOA Connection — Step 1 "Everyone on Stage"

In QAOA, we start by putting all 5 qubits in superposition simultaneously.
This creates ALL possible bitstrings (candidate solutions) at once.

In [ ]:
# ── 4.1  Build and analyze QAOA Step 1 ───────────────────────────────────────
n = 5
qc_step1 = QuantumCircuit(n, n)
qc_step1.h(range(n))    # "Everyone on Stage"
qc_step1.measure(range(n), range(n))

print("QAOA Step 1 Circuit ('Everyone on Stage'):")
print(qc_step1.draw('text'))

In [ ]:
# Run for many shots
compiled  = transpile(qc_step1, simulator)
result    = simulator.run(compiled, shots=3200).result()
counts    = result.get_counts()

# Compute cut value for each observed bitstring
edges = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

def cut_value(bitstring, edges):
    return sum(1 for u,v in edges if bitstring[u] != bitstring[v])

# Aggregate by cut value
cut_total = {}
for bs, cnt in counts.items():
    cv = cut_value(bs, edges)
    cut_total[cv] = cut_total.get(cv, 0) + cnt

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: all bitstrings (sample of first 20)
sample = dict(sorted(counts.items())[:20])
axes[0].bar(range(len(sample)), list(sample.values()), color='steelblue')
axes[0].set_xlabel('Bitstring index (first 20 shown)')
axes[0].set_ylabel('Count')
axes[0].set_title('Measurement counts — uniform superposition')
axes[0].axhline(y=3200/32, color='red', linestyle='--', label=f'Expected = {3200/32:.0f}')
axes[0].legend()

# Right: distribution over cut values
cv_keys = sorted(cut_total.keys())
axes[1].bar(cv_keys, [cut_total[k] for k in cv_keys], color='darkorange')
axes[1].set_xlabel('Cut value')
axes[1].set_ylabel('Total shots')
axes[1].set_title('Cut value distribution — before QAOA\n(should be roughly uniform)')
axes[1].set_xticks(cv_keys)

plt.tight_layout()
plt.show()

print("\nCut value distribution:")
total = sum(cut_total.values())
for cv in sorted(cut_total.keys()):
    pct = 100*cut_total[cv]/total
    print(f"  Cut={cv}: {cut_total[cv]:4d} shots ({pct:.1f}%)")
print("\nGoal of QAOA: concentrate probability at cut=6.")

---
### ✏️ Exercise 2.2 — Explore superposition statistics

1. Run the 5-qubit circuit above with **100 shots**, **1000 shots**, and **10000 shots**.
   - How does the distribution change?
   - How close does each cut value get to its theoretical probability?

2. What is the **theoretical probability** of each cut value 0 through 6?
   - Hint: enumerate all 32 bitstrings and compute their cut values.

3. What is the expected cut value under the uniform distribution?
   (i.e., what is the average cut value if you pick randomly?)

In [ ]:
# ── Part 1: Effect of shot count ─────────────────────────────────────────────
print("=== Effect of Shot Count ===")
for shots in [100, 1000, 10000]:
    r = simulator.run(transpile(qc_step1, simulator), shots=shots).result()
    c = r.get_counts()
    cv6 = sum(cnt for bs, cnt in c.items() if cut_value(bs, edges) == 6)
    print(f"  shots={shots:6d}: P(cut=6) ≈ {cv6/shots:.4f}")

# ── Part 2: Theoretical probabilities ────────────────────────────────────────
print("\n=== Theoretical Probabilities ===")
all_bitstrings = [format(i,'05b') for i in range(32)]
theory = {}
for bs in all_bitstrings:
    cv = cut_value(bs, edges)
    theory[cv] = theory.get(cv, 0) + 1

print(f"  {'Cut':>5} {'Count':>6} {'P(cut)':>8}")
for cv in sorted(theory.keys()):
    print(f"  {cv:>5} {theory[cv]:>6} {theory[cv]/32:>8.4f}")

# ── Part 3: Expected cut value ────────────────────────────────────────────────
expected_cut = sum(cut_value(bs, edges) for bs in all_bitstrings) / 32
print(f"\n=== Expected Cut Value ===")
print(f"  E[cut] = {expected_cut:.4f}")
print(f"  QAOA goal: push this toward {max(cut_value(bs,edges) for bs in all_bitstrings)}")

---
## ✅ Lab 2 Summary

| Concept | Key Insight |
|---------|-------------|
| Qubit state | Vector of length 1: $[a_0, a_1]$ where $|a_0|^2+|a_1|^2=1$ |
| Bloch sphere | Every qubit state = a point on a unit sphere |
| H gate | Rotates from north pole to +x equator: $|0\rangle \to |+\rangle$ |
| Phase vs amplitude | $|+\rangle$ and $|-\rangle$ have identical measurement probs but different phases |
| Pauli-Z | Flips sign of $|1\rangle$ component; 180° rotation around z-axis |
| QAOA Step 1 | H on all qubits = equal superposition of all 32 candidate solutions |

## 🔭 Preview of Lab 3
Next: **Rotation Operators** — the $R_Z(\theta)$ gate that QAOA uses to **encode cost into phase**. This is QAOA Step 2: *Judges' Scores*.

---
*QOS Lab 2 | Prof. Chansu Yu | Cleveland State University*